# QuantJourney SDK - Universe Construction and Liquidity Screen

This notebook demonstrates a QuantJourney SDK workflow that builds an investable equity universe from prices, volumes, optional FMP screener output, TTM ratios and short-interest context.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Market Data Helpers

In [ ]:
def price_frame(symbol: str, start: str=START, end: str=END) -> pd.DataFrame:
    payload = qj.eod.get_historical_prices(symbol=symbol, start_date=start, end_date=end)
    rows = as_rows(payload)
    if not rows and isinstance(unwrap(payload), dict):
        value = unwrap(payload)
        rows = value.get(symbol) or value.get(symbol.upper()) or []
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f'No price data returned for {symbol}')
    df['date'] = pd.to_datetime(df['date'])
    for col in ['open', 'high', 'low', 'close', 'adjusted_close', 'volume']:
        if col in df:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'adjusted_close' in df and df['adjusted_close'].notna().any():
        df['price'] = df['adjusted_close'].fillna(df['close'])
    else:
        df['price'] = df['close']
    if 'volume' not in df:
        df['volume'] = np.nan
    return df.dropna(subset=['price']).sort_values('date').set_index('date')

def price_panel(symbols: list[str], start: str=START, end: str=END) -> tuple[pd.DataFrame, pd.DataFrame]:
    prices = {}
    volumes = {}
    for symbol in symbols:
        df = price_frame(symbol, start=start, end=end)
        prices[symbol] = df['price']
        volumes[symbol] = df['volume']
    return (pd.DataFrame(prices).dropna(how='all'), pd.DataFrame(volumes).reindex(pd.DataFrame(prices).index))

def returns(prices: pd.DataFrame) -> pd.DataFrame:
    return prices.pct_change().replace([np.inf, -np.inf], np.nan).dropna(how='all')

def dollar_adv(prices: pd.DataFrame, volumes: pd.DataFrame, window: int=63) -> pd.DataFrame:
    return (prices * volumes).rolling(window).mean()


In [ ]:
seed_symbols = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'AVGO', 'JPM', 'LLY', 'XOM', 'UNH', 'V']
screener = qj.fmp.get_stock_screener(marketCapMoreThan=10000000000, limit=50)
prices, volumes = price_panel(seed_symbols)
ret = returns(prices)
adv = dollar_adv(prices, volumes).iloc[-1]


In [ ]:
ratio_rows = []
for symbol in seed_symbols:
    payload = qj.fmp.get_financial_ratios_ttm(symbol=symbol)
    data = unwrap(payload) or {}
    ratio_rows.append({'symbol': symbol, 'pe_ttm': pd.to_numeric(data.get('peRatioTTM'), errors='coerce'), 'gross_margin_ttm': pd.to_numeric(data.get('grossProfitMarginTTM'), errors='coerce'), 'fcf_yield_ttm': 1 / pd.to_numeric(data.get('priceToFreeCashFlowRatioTTM'), errors='coerce') if data.get('priceToFreeCashFlowRatioTTM') else np.nan})
ratios = pd.DataFrame(ratio_rows).set_index('symbol')
short_interest = {symbol: qj.finra.get_short_interest(symbol=symbol) for symbol in seed_symbols[:5]}


In [ ]:
features = pd.DataFrame(index=seed_symbols)
features['adv_usd'] = adv.reindex(seed_symbols)
features['volatility_63d'] = ret[seed_symbols].tail(63).std() * np.sqrt(252)
features['momentum_126d'] = prices[seed_symbols].pct_change(126).iloc[-1]
features = features.join(ratios)
features['liquid'] = features['adv_usd'] > 50000000
features['quality_score'] = features['gross_margin_ttm'].rank(pct=True) + features['fcf_yield_ttm'].rank(pct=True)
features['risk_penalty'] = features['volatility_63d'].rank(pct=True)
features['universe_score'] = features['quality_score'] + features['momentum_126d'].rank(pct=True) - features['risk_penalty']
universe = features.query('liquid').sort_values('universe_score', ascending=False)
display(universe)
universe[['adv_usd', 'momentum_126d', 'volatility_63d']].plot(kind='bar', subplots=True, layout=(1, 3), figsize=(15, 4), title='Universe diagnostics')
plt.tight_layout()
plt.show()


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.